# 2. Robust Model Evaluation and Comparison

Robust model evaluation is a cornerstone of responsible machine learning.
Without rigorous evaluation and principled comparison, reported performance metrics can be misleading, non-reproducible, or even unsafe when deployed in real-world settings.

Robust evaluation is imperative to ensure:

- Scientific rigor: results should be reproducible, statistically sound, and not driven by random variation or evaluation artifacts.
- Safer AI applications: models deployed in healthcare, finance, policy, or industry must behave reliably under realistic conditions.

## 2.1 What Does “Robust” Mean?

A robust model maintains stable and reliable performance:

- When exposed to noisy, incomplete, out-of-distribution, or adversarial inputs.

- When deployed in environments different from those seen during training.

- When retrained on slightly different samples of data.

- When compared fairly against alternative modeling approaches.

## 2.2 This Tutorial

In this tutorial, we try to demonstrate the common pitfall and problems associated with model evaluation and comparison. To this end, we are going to use the Diabetes dataset from Scikit-Learn. Following the logics from the last lecture, we are going to cast the problem into a binary classification problem using the average of target variable as the threshold (dividing the patients into low risk 0, and high risk 1 classes).

In [ ]:
# Importing Packages

######## Data Processing Packages ########
import numpy as np
import pandas as pd

######## Visualization Packages ########
import matplotlib.pyplot as plt

######## Data and Preprocessing Modules ########
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

##### Machine Learning Models ########
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC

##### Model Evaluation Modules ########
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate, train_test_split, RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

##### Statistical Tests ########
from scipy.stats import t

######## Utilities ########
from itertools import combinations
from math import factorial


In [ ]:
# Load the dataset
data = load_diabetes(scaled=False)
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)
feature_names = data.feature_names

# From continuous to binary outcomes 
threshold = np.mean(y)  # 
y_binary = (y >= threshold).astype(int)

## 2.3 The Illusion of Performance

A common pitfall is using a single **train–test split** iteration with a specific random seed for model evaluation and comparison. This produces a **single point estimate** of performance. That estimate can be fragile for two reasons:

1. **The Bias of “Luck”**: the split is one arbitrary sample of the data. A model may look strong simply because the test set happened to be “easy” (optimistic bias).

2. **The Hidden Variance**: even if we report an average score (e.g., K-fold CV), we often ignore **how much the score fluctuates** across different training samples. Without quantifying this variance, the **reliability** of the estimate is unknown.

**Bottom line:** a single score is a *snapshot of luck*. If the **winner depends on the random seed**, the comparison is **not reliable**. A robust evaluation should report a *distribution* of performance (uncertainty), not a single score.

This motivates robust bias estimation and explicit accounting for model variance.

In [ ]:
random_state = 42 # 42 1 100

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_binary, test_size=0.2,stratify=y_binary, random_state=random_state)

# Create, fit, and predict using the Gradient Boosting Classifier model
gbc = GradientBoostingClassifier(random_state=random_state)
gbc.fit(X_train, y_train)
y_pred_gbc = gbc.predict_proba(X_test)


# Create, fit, and predict using the Linear Support Vector Classifier model
svc = SVC(kernel="linear", probability=True, random_state=random_state)
svc.fit(X_train, y_train)
y_pred_svc = svc.predict_proba(X_test)

# Calculate accuracy score
auc_gbc = roc_auc_score(y_test, y_pred_gbc[:, 1])  
auc_svc = roc_auc_score(y_test, y_pred_svc[:, 1])  

print("SVC AUC:", auc_svc)
print("GBC AUC:", auc_gbc)

## 2.4 Solution 1: Repeated train–test splits (Monte Carlo cross-validation)

Repeated train–test splits (Monte Carlo cross-validation) repeatedly create random training and test partitions, train the model on each split, and record performance. By averaging across repetitions, we approximate expected performance, while the variability across splits reflects sensitivity to sampling. However, this approach is less data-efficient and the splits are not fully independent.

In [ ]:
n_repeats = 10 #100
test_size = 0.2

auc_gbc_list = []
auc_svc_list = []

for seed in range(n_repeats):

    # Repeated random split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_binary,
        test_size=test_size,
        random_state=seed,
        stratify=y_binary
    )

    # Gradient Boosting
    gbc = GradientBoostingClassifier(random_state=seed)
    gbc.fit(X_train, y_train)
    y_pred_gbc = gbc.predict_proba(X_test)[:, 1]

    # Linear SVC
    svc = SVC(kernel="linear", probability=True, random_state=seed)
    svc.fit(X_train, y_train)
    y_pred_svc = svc.predict_proba(X_test)[:, 1]

    # Compute AUC
    auc_gbc = roc_auc_score(y_test, y_pred_gbc)
    auc_svc = roc_auc_score(y_test, y_pred_svc)

    auc_gbc_list.append(auc_gbc)
    auc_svc_list.append(auc_svc)

# Convert to arrays
auc_gbc_array = np.array(auc_gbc_list)
auc_svc_array = np.array(auc_svc_list)

# Summary statistics
print("Repeated Train-Test Split Results")
print("-----------------------------------")
print(f"GBC Mean AUC: {auc_gbc_array.mean():.3f} ± {auc_gbc_array.std():.3f}")
print(f"SVC Mean AUC: {auc_svc_array.mean():.3f} ± {auc_svc_array.std():.3f}")


#  Boxplot Visualization
plt.figure(figsize=(7, 5))

box = plt.boxplot(
    [auc_svc_array, auc_gbc_array],
    labels=["SVC", "GBC"],
    patch_artist=True,
    showmeans=True,
    meanline=True,
    widths=0.6
)


# Style lines
for element in ["whiskers", "caps", "medians", "means"]:
    plt.setp(box[element], linewidth=2)

plt.ylabel("AUC", fontsize=12)
plt.title("Model Performance Distribution\nRepeated Train-Test Splits", fontsize=13)

plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.ylim(min(min(auc_svc_array), min(auc_gbc_array)) - 0.01,
         max(max(auc_svc_array), max(auc_gbc_array)) + 0.01)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()



**Exercise 1:** Adapt the code to also accommodate the grid search hyperparameter optimization in the inner loop. 

## 2.5 Solution 2: Repeated nested K-fold cross-validation

Repeated nested K-fold cross-validation separates hyperparameter tuning and performance estimation. In the inner loop, hyperparameters are optimized, while the outer loop provides an unbiased estimate of generalization performance. Repeating the procedure multiple times yields a distribution of performance estimates. Although this controls selection bias and uses data efficiently, the variance estimate is typically downward biased due to overlapping folds and dependency between splits.

In [ ]:
# Identify numerical and categorical columns
X['sex'] = X['sex'].astype('object') # just for demonstration I change the datatype of sex into object
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object','bool']).columns


# Preprocessing for numerical data: imputation + scaling
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Preprocessing for categorical data: imputation + one-hot encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing steps fpr no-tree models
preprocessor_non_tree = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough'
)

# Combine preprocessing steps for tree-based models
preprocessor_tree = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_cols), # I can do this as I know I have no missing values
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='passthrough'
)


# For illustrative purpose we use the minimal set of hyperparameters
models = {
    "Support Vector Machine": {
        "model": SVC(kernel="linear", probability=True, random_state=seed),
        "params": {
            "model__C": [0.1, 1.0, 10.0],
        }
    },
    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=random_state),
        "params": {
            "model__n_estimators": [50, 100, 200],
        }
    }
}

In [ ]:
outer_split_num = 5
inner_split_num = 5

random_state = 42   # When using repeated outer loop, we may set this to None.

#outer_cv = StratifiedKFold(n_splits=outer_split_num, shuffle=True, random_state=random_state)
outer_cv = RepeatedStratifiedKFold(n_repeats=10, n_splits=outer_split_num,random_state=random_state)

# Inner CV: hyperparameter optimization
inner_cv = StratifiedKFold(n_splits=inner_split_num, shuffle=True, random_state=random_state)


results = {}

for name, spec in models.items():
    
    # Build full pipeline based on type of classifiers
    if name == "Gradient Boosting": # We can also check the type of model using isinstance() function. --- IGNORE ---
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor_tree),
            ("model", spec["model"])
        ]) 
    else:    
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor_non_tree),
            ("model", spec["model"])
        ])
        
    # Grid search (inner loop)
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=spec["params"],
        cv=inner_cv,
        scoring="roc_auc" # We can use only one metric for hyperparameter tuning. Be careful when choosing one.
    )
    
    # Nested CV (outer loop)
    cv_results = cross_validate(
        estimator=grid_search,
        X=X,
        y=y_binary,
        cv=outer_cv,
        scoring=["f1", "roc_auc","precision","recall"],
        return_estimator=True # this is useful for model diagnostic and explanation later.
    )
    
    results[name] = cv_results

    print(f"\n{name}")

    print(f"  F1-Score: {np.mean(results[name]['test_f1']):.3f} ± {np.std(results[name]['test_f1']):.3f}")
    print(f"  ROC-AUC:  {np.mean(results[name]['test_roc_auc']):.3f} ± {np.std(results[name]['test_roc_auc']):.3f}")
    print(f"  Precision: {np.mean(results[name]['test_precision']):.3f} ± {np.std(results[name]['test_precision']):.3f}")
    print(f"  Recall: {np.mean(results[name]['test_recall']):.3f} ± {np.std(results[name]['test_recall']):.3f}")

## 2.6 Solution 3: Bootstrapping

Bootstrap resampling repeatedly samples the training data with replacement, trains the model on each bootstrap sample, and evaluates it either on out-of-bag observations or on a fixed test set. This procedure approximates the sampling distribution of performance and can be used to estimate optimism and prediction variability. However, in small datasets it may distort the underlying data structure.

In [ ]:
n_bootstraps = 10 #100 If large enough we do not even need to set random state.

random_state = None

rng = np.random.default_rng(random_state)

n = len(y_binary)  # works for Series/array
indices = np.arange(n)

auc_gbc_list = []
auc_svc_list = []
n_skipped = 0

# Helper: robust row indexing for pandas or numpy
def row_select(A, idx):
    return A.iloc[idx] if hasattr(A, "iloc") else A[idx]

for b in range(n_bootstraps):
    # Bootstrap sample (with replacement)
    boot_idx = rng.choice(indices, size=n, replace=True)

    # Out-of-bag (OOB) indices = those not selected
    oob_mask = np.ones(n, dtype=bool)
    oob_mask[boot_idx] = False
    oob_idx = indices[oob_mask]

    # Need enough OOB samples + both classes to compute AUC
    if oob_idx.size < 10:
        n_skipped += 1
        continue

    y_oob_tmp = row_select(y_binary, oob_idx)
    # ensure we can check unique values
    y_oob_vals = np.asarray(y_oob_tmp)
    if np.unique(y_oob_vals).size < 2:
        n_skipped += 1
        continue

    X_boot = row_select(X, boot_idx)
    y_boot = row_select(y_binary, boot_idx)

    X_oob = row_select(X, oob_idx)
    y_oob = y_oob_tmp

    # Train models on bootstrap sample
    gbc = GradientBoostingClassifier(random_state=random_state)
    gbc.fit(X_boot, y_boot)
    gbc_scores = gbc.predict_proba(X_oob)[:, 1]

    svc = SVC(kernel="linear", probability=True, random_state=random_state)
    svc.fit(X_boot, y_boot)
    svc_scores = svc.predict_proba(X_oob)[:, 1]

    # Evaluate on OOB set
    auc_gbc_list.append(roc_auc_score(y_oob, gbc_scores))
    auc_svc_list.append(roc_auc_score(y_oob, svc_scores))

auc_gbc = np.array(auc_gbc_list)
auc_svc = np.array(auc_svc_list)

print("Bootstrap (OOB) Results")
print("------------------------")
print(f"Bootstraps attempted: {n_bootstraps}")
print("------------------------")
print(f"GBC Mean AUC: {auc_gbc.mean():.3f} ± {auc_gbc.std():.3f}")
print(f"SVC Mean AUC: {auc_svc.mean():.3f} ± {auc_svc.std():.3f}")

# Boxplot 
plt.figure(figsize=(7, 5))

box = plt.boxplot(
    [auc_svc, auc_gbc],
    labels=["SVC (OOB)", "GBC (OOB)"],
    patch_artist=True,
    showmeans=True,
    meanline=True,
    widths=0.6
)

colors = ["#4C72B0", "#55A868"]
for patch, color in zip(box["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)

for element in ["whiskers", "caps", "medians", "means"]:
    plt.setp(box[element], linewidth=2)

rng_vis = np.random.default_rng(0)
jitter_strength = 0.05
x_positions = [1, 2]

for x0, vals in zip(x_positions, [auc_svc, auc_gbc]):
    jitter = rng_vis.normal(0, jitter_strength, size=len(vals))
    plt.scatter(
        np.full(len(vals), x0) + jitter,
        vals,
        alpha=0.6,
        s=35
    )

plt.ylabel("AUC", fontsize=12)
plt.title("Bootstrap (OOB) AUC Distribution", fontsize=13)
plt.grid(axis="y", linestyle="--", alpha=0.4)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

## 2.7 Robust Model Comparison: Theory and Motivation

### 2.7.1 Why Model Comparison Requires Statistical Testing

When comparing machine learning models, differences in performance are often small. However, performance estimates obtained through cross-validation are **random variables**, not fixed quantities.

A difference such as $\hat{R}_A - \hat{R}_B = 0.01$ may arise from:
- A true underlying performance difference, or
- Random fluctuations due to data sampling.

Therefore, robust model comparison must answer:

> Is the observed difference larger than what we would expect from sampling variability alone?

This requires statistical inference.

###  2.7.2 The Challenge: Correlated Performance Estimates

In nested K-fold cross-validation:

- Each model is evaluated on the same outer splits.
- Training sets overlap substantially across folds.
- Performance estimates across folds are therefore **not independent**.

Classical paired t-tests assume independence of observations. This assumption is violated in cross-validation settings.

If ignored, this leads to:
- Underestimated variance
- Inflated Type I error (false positives)
- Overconfident conclusions

### 2.7.3 Repeated Resampling as an Evaluation Framework

Robust model comparison requires **multiple, paired performance estimates** obtained under repeated data perturbations. This can be achieved through:

- Repeated train–test splits (Monte Carlo cross-validation)
- Repeated K-fold cross-validation
- Nested cross-validation (when hyperparameter tuning is involved)

In all cases, the procedure generates multiple evaluations of each model:

$\hat{R}_{A,1}, \dots, \hat{R}_{A,m}$

$\hat{R}_{B,1}, \dots, \hat{R}_{B,m}$

Each pair $(\hat{R}_{A,i}, \hat{R}_{B,i})$ is obtained from the **same data split**, ensuring a paired comparison.

We define split-wise differences:

$d_i = \hat{R}_{A,i} - \hat{R}_{B,i}$

Statistical comparison is then performed on the distribution of these paired differences.

#### Special Case: Nested Cross-Validation

When hyperparameter tuning is involved, nested cross-validation ensures that:

- Model selection is performed in the inner loop.
- Performance estimation is performed in an outer loop.
- The outer-loop scores provide unbiased estimates of generalization performance.

In this setting, inference is performed on the outer-loop paired differences.


### 2.7.4 Corrected Paired t-Test (Nadeau & Bengio, 2003)

To account for correlation between folds, we use a **corrected paired t-test**. The classical paired t-statistic is:

$t = \frac{\bar{d}}{s_d / \sqrt{m}}$

where:
- $\bar{d}$ is the mean difference
- $s_d^2$ is the sample variance of differences
- $m$ is the number of outer evaluations

However, cross-validation induces correlation between folds. Nadeau & Bengio proposed a variance correction:

$\mathrm{Var}_{corr} = s_d^2 \left(\frac{1}{m} + \frac{n_{test}}{n_{train}}\right)$

The corrected t-statistic becomes:

$t_{corr} = \frac{\bar{d}}{\sqrt{\mathrm{Var}_{corr}}}$

This adjustment inflates the variance estimate to compensate for dependency between folds, leading to more reliable inference.

Then for two models A and B:

Null hypothesis:
$H_0: \mathbb{E}[R_A - R_B] = 0$

Alternative hypothesis:
$H_1: \mathbb{E}[R_A - R_B] \neq 0$

If the corrected p-value is small (e.g., < 0.05), we reject the null hypothesis and conclude that the performance difference is unlikely due to sampling variability alone.

When comparing more than two models, the number of pairwise tests grows as: $\binom{M}{2}$. Performing multiple tests increases the probability of false discoveries. To control the family-wise error rate, we apply **Bonferroni correction**:

$p_{corrected} = \min(1, p \times \text{number of comparisons})$

This provides a conservative but safe adjustment.


In [ ]:
# The code is adapted from sklearn tutorial at 
# https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_stats.html

def corrected_ttest(differences, n_train, n_test):
    """
    Nadeau & Bengio corrected paired t-test for correlated estimates.

    differences: array of paired differences (model_i - model_k) across outer splits
    n_train, n_test: sizes of training/test sets in the outer loop (approx constants for K-fold)
    """
    differences = np.asarray(differences)
    m = len(differences)
    df = m - 1

    mean_diff = np.mean(differences)
    var_diff = np.var(differences, ddof=1)

    # Corrected variance (Nadeau & Bengio)
    corrected_var = var_diff * (1.0 / m + (n_test / n_train))
    corrected_std = np.sqrt(corrected_var)

    t_stat = mean_diff / corrected_std
    # two-sided p-value
    p_val = 2 * t.sf(np.abs(t_stat), df)
    return t_stat, p_val


def pairwise_corrected_ttests_nested_cv(
    results,
    y,
    metric="roc_auc",
    outer_splits=5,
    bonferroni=True
):
    """
    Pairwise frequentist comparisons across multiple models using corrected t-tests.

    results: dict produced by your loop, each entry contains arrays like 'test_roc_auc'
    y: target array/Series (needed for n)
    metric: one of {"roc_auc","f1","precision","recall"} matching keys 'test_{metric}'
    outer_splits: K in outer K-fold (used to approximate n_train/n_test)
    bonferroni: whether to apply Bonferroni correction

    Returns:
      pairwise_comp_df: DataFrame with pairwise t-stats and (corrected) p-values
      model_scores: DataFrame with outer-split scores per model (rows=models, cols=split index)
    """
    key = f"test_{metric}"

    # Build model x outer-split score matrix
    model_names = list(results.keys())
    model_scores = pd.DataFrame(
        {name: np.asarray(results[name][key]) for name in model_names}
    ).T  # rows=models, cols=outer evaluations

    # Sanity check: all models must have same number of outer evaluations
    n_evals = model_scores.shape[1]
    if model_scores.isna().any().any():
        raise ValueError("Found NaNs in model scores. Check CV outputs.")
    if len({len(results[name][key]) for name in model_names}) != 1:
        raise ValueError("Models have different numbers of outer scores; ensure same outer CV splits.")

    # Approximate n_train/n_test for outer K-fold
    n = len(y)
    n_test = n / outer_splits
    n_train = n - n_test

    # Number of pairwise comparisons
    n_models = len(model_scores)
    n_comparisons = factorial(n_models) / (factorial(2) * factorial(n_models - 2))

    pairwise_rows = []
    for i, k in combinations(range(n_models), 2):
        scores_i = model_scores.iloc[i].values
        scores_k = model_scores.iloc[k].values
        differences = scores_i - scores_k

        t_stat, p_val = corrected_ttest(differences, n_train=n_train, n_test=n_test)

        if bonferroni:
            p_val = p_val * n_comparisons
            p_val = 1.0 if p_val > 1.0 else p_val

        pairwise_rows.append([
            model_scores.index[i],
            model_scores.index[k],
            np.mean(scores_i),
            np.mean(scores_k),
            np.mean(differences),
            t_stat,
            p_val
        ])

    pairwise_comp_df = pd.DataFrame(
        pairwise_rows,
        columns=["model_1", "model_2", "mean_1", "mean_2", "mean_diff(1-2)", "t_stat", "p_val"]
    ).round(4)

    return pairwise_comp_df, model_scores


# -------------------------
# Example usage
# -------------------------
pairwise_auc_df, auc_scores = pairwise_corrected_ttests_nested_cv(
    results=results,
    y=y_binary,
    metric="roc_auc",
    outer_splits=outer_split_num,
    bonferroni=True
)

pairwise_auc_df

Besides the p-value and effect size, always report the t-stat. It tells us how many standard errors away from zero the observed difference is.

For example:
- t=0 → no difference
- t=1 → difference = 1 standard error
- t=2 → difference = 2 standard errors

The larger $|t|$, the stronger the evidence against the null hypothesis is.


<div style="
    display: flex;
    align-items: center;
    justify-content: flex-end;
    gap: 8px;
    font-size: 9px;
    color: #888;
    font-style: italic;
    margin-top: 30px;
">
    <span> © 2026 S.M. Kia (Tilburg University) — Licensed under the Creative Commons Attribution 4.0 International (CC BY 4.0). This notebook is provided for educational purposes. Redistribution and reuse are permitted with proper attribution.</span>
</div>
